# Meta: Baseline Evaluations

In [ ]:
# Imports
import os, json, sys
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
import numpy as np
import seaborn as sns
import language_tool_python as ltp
from typing import List, Dict, Any

In [2]:
# Paths
notebook_path = os.getcwd()
project_root = os.path.dirname(os.path.abspath(notebook_path))
data_dir = os.path.join(project_root, "data")
evaluations_dir = os.path.join(data_dir, "evaluations")
responses_dir = os.path.join(data_dir, "generated")
vocab_dir = os.path.join(data_dir, "vocab")
db_path = os.path.join(data_dir, "DB.db")


In [3]:
# SQL Alchemy session
engine = create_engine(f"sqlite:///{db_path}")
session_factory = sessionmaker(bind=engine)
session = session_factory()

In [4]:
# Helper functions
def get_prompt_eval_dir(prompt_id):
    return os.path.join(evaluations_dir, f"prompt{prompt_id}")

def get_unique_ids(prompt_eval_dir):
    unique_ids = []
    for filename in os.listdir(prompt_eval_dir):
        if filename.endswith(".json"):
            unique_ids.append(filename.split(".json")[0])
    return unique_ids

def load_evaluation(prompt_id):
    prompt_dir = get_prompt_eval_dir(prompt_id)
    unique_ids = get_unique_ids(prompt_dir)
    res = []
    for unique_id in unique_ids:
        file_path = os.path.join(prompt_dir, f"{unique_id}.json")
        with open(file_path, "r", encoding="utf-8") as file:
            res.append(json.load(file))
    return res

def load_evaluations(prompt_ids):
    res_list = []
    for prompt_id in prompt_ids:
        res_list.append(load_evaluation(prompt_id))
    return res_list

In [5]:
# Ground truth
gt_id = "gt.45af4a46-d238-4251-b734-d407602299bd"
gtb_id = "gtb.45af4a46-d258-4251-b734-d407602299bd"

gt_eval_path = os.path.join(evaluations_dir, f"{gt_id}.json")
gtb_eval_path = os.path.join(evaluations_dir, f"{gtb_id}.json")

gt_path = os.path.join(responses_dir, f"{gt_id}.json")
gtb_path = os.path.join(responses_dir, f"{gtb_id}.json")

with open(gt_path, "r", encoding="utf-8") as file:
    gt_res = json.load(file)
with open(gt_path, "r", encoding="utf-8") as file:
    gtb_res = json.load(file)

In [106]:
Befunds = [
    # Wrong Befund
    "Letzte Voruntersuchung des Halses vom zweiten 2424.letzte Voruntersuchung des Thorax vom 14.6.2024.\n\nHals:\nDie vorbekannten, suspekten cervicalen Lymphknoten stellen sich im Vergleich zur den Voruntersuchung hypodenser und dar, exemplarisch rechts supraclaviculär bis 0,6 cm quer (VU: 0,9 cm quer) messend. Ansonsten regelrechte, symmetrische Darstellung der Halsweichteile ohne Nachweis einer umschriebenen Raumforderung. Weitgehend symmetrische Anlage der normal großen Schilddrüse. Regelrechte Kontrastierung der Halsgefäße. Regelrechte Belüftung der partiell miterfassten Nasennebenhöhlen.\n\nThorax:\nMultiple polytope malignomsuspekte Lungenrundherde beidseits, exemplarisch der größte im posterioren Oberlappen rechts, 2,3 x 1,9 cm axial messend, im kurzfristigen Verlauf größenkonstant. Keine neu zur Voruntersuchung abgrenzbaren malignomsuspekten intrapulmonalen Rundherde. Beidseits dorsobasal betonte dystelektatische Veränderungen, linksseitig betont. Die vorbekannten, pathologisch vergrößerten Lymphknoten sind insbesondere apikal im Vergleich zur Voruntersuchung hypodenser abgrenzbar und mit weiterhin geringer randständiger Kontrastmittel-Aufnahme. Rechts hilär und infrakarinal sind die bekannten Lymphknoten weiterhin weichteildicht abgrenzbar und ebenfalls mit randständiger Kontrastmittel-Aufnahme. Kein Pleuraerguss. Normgroßes Herz ohne Dekompensationszeichen. Kein Perikarderguss. Regelrechte Darstellung der großen intrathorakalen Gefäße. \nLinks axillär zeigt sich neu zur Voruntersuchung ein rundlich konfigurierter, größenprogredienter Lymphknoten ohne pathologische Vergrößerung, bis 0,7 cm quer messend.\n\nMiterfasstes oberes Abdomen:\nZahlreiche polytope, disseminierte, hypodense Leberläsionen ohne relevante Kontrastmittelanreicherung in der aktuellen Kontrastmittelphase, die größte exemplarisch ventral angrenzend an die linke Lebervene, 2,4 cm im Durchmesser, im Vergleich zur Voruntersuchung zunehmend hypodenser und tendenziell größenregredient. Mehrere disseminiert verteilte Milzläsionen, die größte exemplarisch subkapsulär, 2,4 cm im Durchmesser, im Vergleich zur Voruntersuchung ebenfalls hypodenser als Hinweis auf ein gutes Therapie-Ansprechen. Winzige hypodense Nierenzysten links. Ansonsten unauffällige Darstellung der partiell miterfassten parenchymatösen Oberbauchorgane.\n\nMiterfasstes Skelett:\nVorbekannte, ossäre Metastasen in BWK 10 und BWK 12 mit vorbekannten pathologischen Frakturen, jedoch ohne relevante Nachsinterung. Die ossäre Metastase in BWK 10 zeigt im Vergleich zur Voruntersuchung aktuell keine relevante Weichteilkomponente mehr. Keine zwischenzeitlich neu abgrenzbaren ossären Läsionen. Keine neue, frischen Frakturen.\n",
    # Correct Befund
    "Letzte Voruntersuchung des Halses vom 02.04.24. Letzte Voruntersuchung des Thorax vom 14.06.2024.\n\nHals:\nDie vorbekannten, suspekten zervikalen Lymphknoten stellen sich im Vergleich zur Voruntersuchung hypodenser und größenregredient dar, exemplarisch rechts supraclaviculär bis 0,6 cm quer (VU: 0,9 cm quer) messend. Ansonsten regelrechte, symmetrische Darstellung der Halsweichteile ohne Nachweis einer umschriebenen Raumforderung. Weitgehend symmetrische Anlage der normal großen Schilddrüse. Regelrechte Kontrastierung der Halsgefäße. Regelrechte Belüftung der partiell miterfassten Nasennebenhöhlen. Bezüglich der ehemaligen paravertebralen Raumforderung siehe unten (Abschnitt \"Skelett\").\n\nThorax:\nMultiple polytope malignomsuspekte Lungenrundherde beidseits, exemplarisch der größte im posterioren Oberlappen rechts, 1,3 x 1,2 cm axial messend, im kurzfristigen Verlauf größenkonstant. Keine neu zur Voruntersuchung abgrenzbaren malignomsuspekten intrapulmonalen Rundherde. Beidseits dorsobasal betonte dystelektatische Veränderungen, linksseitig betont. Mediastinale Lymphadenopathie regredient; exemplarisch präkarinaler Lymphknoten regredient von 2,2 auf nun 1,5 cm quer, sowie zunehmend hypodens als Zeichen des Therapieansprechens. Keine axilläre Lymphadenopathie. Geringer Pleuraerguss rechts. Bds. dorsobasale Belüftungsstörungen. Normal großes Herz ohne Dekompensationszeichen. Kein Perikarderguss. Regelrechte Darstellung der großen intrathorakalen Gefäße. \n\nMiterfasstes oberes Abdomen:\nZahlreiche polytope, disseminierte, hypodense Leberläsionen ohne relevante Kontrastmittelanreicherung in der aktuellen Kontrastmittelphase, die größte exemplarisch im Lebersegment IVa bis 1,6 cm, idem. Mehrere disseminiert verteilte Milzläsionen, exemplarisch bis 1,5 cm messend am Milzoberpol, idem. Bildeindruck der Hepatosplenomegalie mit Kissingphänomen.\n\nMiterfasstes Skelett:\nVorbekannte, ossäre Metastasen in BWK 10 und BWK 12 mit vorbekannten pathologischen Frakturen, jedoch ohne zwischenzeitliche, relevante Nachsinterung (soweit bei teilerfassten BWK 12 eingeschränkt beurteilbar). Die ossäre Metastase in BWK 10 zeigt im Vergleich zur Voruntersuchung aktuell keine relevante Weichteilkomponente mehr. Keine zwischenzeitlich neu abgrenzbaren ossären Läsionen. Keine neuen, frischen Frakturen. Angrenzend der unteren HWS rechts zeigt sich die vormalige hier kontrastmittelaufnehmende Raumforderung erheblich größenregredient im Verlauf (vergleiche CT vom 14.03.2024); aktuell kein relevanter Restbefund. Bekanntermaßen multiple, zwischenzeitlich demaskierte, unter Therapie mehrsklerosierte, in erster Linie metastasensuspekte Läsionen, exemplarisch in BWK 6 bis 0,6 cm.",
    # Model Befund
    "Letzte Voruntersuchung des Halses vom 14.6.2024, letzte Voruntersuchung des Thorax vom 14.6.2024.\n\nHals:\nDie bekannten suspekten cervicalen Lymphknoten stellen sich im Vergleich zur Voruntersuchung hypodenser dar, exemplarisch rechts supraclaviculär bis 0,6 cm quer (VU: 0,9 cm quer) messend. Ansonsten regelmäßige, symmetrische Darstellung der Halsweichteile ohne Nachweis einer umschriebenen Raumforderung. Weitgehend symmetrische Anlage der normal großen Schilddrüse. Regelrechte Kontrastierung der Halsgefäße. Regelrechte Belüftung der partiell miterfassten Nasennebenhöhlen.\n\nThorax:\nMultiple polytope malignomsuspekten Lungenrundherde beidseits, exemplarisch der größte im posterioren Oberlappen rechts, 2,3 x 1,9 cm axial messend, im kurzfristigen Verlauf größenkonstant. Keine neu zur Voruntersuchung abgrenzbaren malignomsuspekten intrapulmonalen Rundherde. Beidseits dorsobasal betonte dystelektatische Veränderungen, linksseitig betont. Die bekannten, pathologisch vergrößerten Lymphknoten sind insbesondere apikal im Vergleich zur Voruntersuchung hypodenser abgrenzbar und mit weiterhin geringer randständiger Kontrastmittel-Aufnahme. Rechts hilär und infrakarinal sind die bekannten Lymphknoten weiterhin weichteildicht abgrenzbar und ebenfalls mit randständiger Kontrastmittel-Aufnahme. Kein Pleuraerguss. Normgroßes Herz ohne Dekompensationszeichen. Kein Perikarderguss. Regelrechte Darstellung der großen intrathorakalen Gefäße. Links axillär zeigt sich neu zur Voruntersuchung ein rundlich konfigurierter, größenprogredienter Lymphknoten ohne pathologische Vergrößerung, bis 0,7 cm quer messend.\n\nMiterfasstes oberes Abdomen:\nZahlreiche polytope, disseminierte, hypodense Leberläsionen ohne relevante Kontrastmittelanreicherung in der aktuellen Kontrastmittelphase, die größte exemplarisch ventral angrenzend an die linke Lebervene, 2,4 cm im Durchmesser, im Vergleich zur Voruntersuchung zunehmend hypodenser und tendenziell größenregredient. Mehrere disseminiert verteilte Milzläsionen, die größte exemplarisch subkapsulär, 2,4 cm im Durchmesser, im Vergleich zur Voruntersuchung ebenfalls hypodenser als Hinweis auf ein gutes Therapie-Ansprechen. Winzige hypodense Nierenzysten links. Ansonsten unauffällige Darstellung der partiell miterfassten parenchymatösen Oberbauchorgane.\n\nMiterfasstes Skelett:\nVorbekannte, ossäre Metastasen in BWK 10 und BWK 12 mit vorbekannten pathologischen Frakturen, jedoch ohne relevante Nachsinterung. Die ossäre Metastase in BWK 10 zeigt im Vergleich zur Voruntersuchung aktuell keine relevante Weichteilkomponente mehr. Keine zwischenzeitlich neu abgrenzbaren ossären Läsionen. Keine neue, frische Fraktur."
]

In [90]:
from nltk.tokenize import sent_tokenize

wrong_sentences = sent_tokenize(Befunds[0], language='german')
correct_sentences = sent_tokenize(Befunds[1], language='german')
model_sentences = sent_tokenize(Befunds[2], language='german')

In [ ]:
from sentence_transformers import SentenceTransformer

model1 = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
model2 = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')

sentences = []

for wrong_sentence, model_sentence in zip(wrong_sentences, model_sentences):
    sentences.append(wrong_sentence)
    sentences.append(model_sentence)

sentence_embeddings1 = model1.encode(sentences, normalize_embeddings=True)
sentence_embeddings2 = model2.encode(sentences, normalize_embeddings=True)

In [109]:
befund_embeddings1 = model1.encode([Befunds[0], Befunds[1]], normalize_embeddings=True)
befund_embeddings2 = model2.encode([Befunds[0], Befunds[1]], normalize_embeddings=True)

In [ ]:
for i in range(0, len(sentences) - 1, 2):
    print(
        sentences[i],
        sentences[i+1],
        '\n',
        (sentence_embeddings1[i] @ sentence_embeddings1[i+1]).item(),
        (sentence_embeddings2[i] @ sentence_embeddings2[i+1]).item(),
        '\n',
        '-'*50
    )



Letzte Voruntersuchung des Halses vom zweiten 2424.letzte Voruntersuchung des Thorax vom 14.6.2024. Letzte Voruntersuchung des Halses vom 14.6.2024, letzte Voruntersuchung des Thorax vom 14.6.2024. 
 0.941860556602478 0.9570621252059937 
 --------------------------------------------------
Hals:
Die vorbekannten, suspekten cervicalen Lymphknoten stellen sich im Vergleich zur den Voruntersuchung hypodenser und dar, exemplarisch rechts supraclaviculär bis 0,6 cm quer (VU: 0,9 cm quer) messend. Hals:
Die bekannten suspekten cervicalen Lymphknoten stellen sich im Vergleich zur Voruntersuchung hypodenser dar, exemplarisch rechts supraclaviculär bis 0,6 cm quer (VU: 0,9 cm quer) messend. 
 0.9902088642120361 0.9876620173454285 
 --------------------------------------------------
Ansonsten regelrechte, symmetrische Darstellung der Halsweichteile ohne Nachweis einer umschriebenen Raumforderung. Ansonsten regelmäßige, symmetrische Darstellung der Halsweichteile ohne Nachweis einer umschriebenen 

np.float32(0.9418607)

In [112]:
# calculate the mean of all cosine similarities
cosine_similarities1 = [
    wrong_sentence @ model_sentence
    for wrong_sentence, model_sentence in zip(sentence_embeddings1[::2], sentence_embeddings1[1::2])
]
cosine_similarities2 = [
    wrong_sentence @ model_sentence
    for wrong_sentence, model_sentence in zip(sentence_embeddings2[::2], sentence_embeddings2[1::2])
]
print((befund_embeddings1[0] @ befund_embeddings1[1]).item(), (befund_embeddings2[0] @ befund_embeddings2[1]).item())
print(np.mean(cosine_similarities1), np.mean(cosine_similarities2))


0.8372658491134644 0.9518904685974121
0.99379927 0.99629325


## Whole Reports

## Befunds